In [ ]:
## This script is for model training using the basic framework and ResNet-50 features

In [ ]:
# import packages
import os, cv2
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

## check GPU
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Current GPU: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

In [ ]:
# load data (first part)
## load shadow-free images
path_wd = '../' ## set working directory
img = cv2.imread(path_wd + 'output/images/shadow_free.jpg')
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) ## transform color channel to RGB
## load LFDP ground-based labels
df = pd.read_csv(path_wd + 'data/labels/LFDP_labels.csv', index_col=0)

In [ ]:
# choose parameters to filter the ground labels
THRESH_DIAM = 20 ## diameter threshold
df = df[df.ALIVE == 'A']
df = df[df.DIAM > THRESH_DIAM]
df = df[df.pix_1 < 9600] ## column range
df = df[df.pix_2 < 15000] ## row range
df.index = range(df.shape[0])

In [ ]:
# create training set
## resolution 
rs = 100
n_row = img.shape[0] // rs
n_col = img.shape[1] // rs
print(n_row, n_col)
## species list
sp_dict = {'PREMON': ['PREMON', 'ROYBOR'], 'CECSCH': ['CECSCH'], 'MANBID': ['MANBID']}
## create data list
x_mat = []
y_mat = []
## dictionary for location map
loc_dict = {}
## loop over the LFDP labels
for i in range(df.shape[0]):
    loc_1 = df['pix_2'][i] // rs
    loc_2 = df['pix_1'][i] // rs
    loc_id = loc_1 * n_col + loc_2
    if loc_id in loc_dict.keys():
        idd = loc_dict[loc_id]
    else:
        idd = len(y_mat)
        rr_1 = loc_1 * rs
        rr_2 = rr_1 + rs
        cc_1 = loc_2 * rs
        cc_2 = cc_1 + rs
        x_mat.append(img[rr_1:rr_2, cc_1:cc_2])
        y_mat.append([0] * 3)
        loc_dict[loc_id] = idd
    ## assign the labels
    if df['SPECIES'][i] in ['PREMON', 'ROYBOR']:
        y_mat[idd][0] = 1
    elif df['SPECIES'][i] in ['CECSCH']:
        y_mat[idd][1] = 1
    elif df['SPECIES'][i] in ['MANBID']:
        y_mat[idd][2] = 1
## transform the list into numpy array
x_mat = np.array(x_mat)
y_mat = np.array(y_mat)

In [ ]:
# optional: rebalance the data set
print('Number of palm patches:', np.sum(y_mat[:, 0]))
print('Number of cecropia patches:', np.sum(y_mat[:, 1]))
print('Number of total patches:', y_mat.shape[0])
REBALANCE = True ## whether rebalance the patches
if REBALANCE ==  True:
    x_list = []
    y_list = []
    for i in range(y_mat.shape[0]):
        if y_mat[i, 0] == 1:
            for j in range(30):
                x_list.append(x_mat[i])
                y_list.append(y_mat[i])
        elif y_mat[i, 1] == 1:
            for j in range(7):
                x_list.append(x_mat[i])
                y_list.append(y_mat[i])
        else:
            x_list.append(x_mat[i])
            y_list.append(y_mat[i])
    x_dat = np.array(x_list)
    y_dat = np.array(y_list)
else:
    x_dat = x_mat
    y_dat = y_mat

In [ ]:
# build the model
## Define Dataset class
class TreePatchDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        """
        images: numpy array [N, 100, 100, 3]
        labels: numpy array [N, 3]
        """
        self.images = images
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        image = self.images[idx]  # [100, 100, 3]
        label = self.labels[idx]  # [3]
        
        if self.transform:
            # Convert to PIL Image then apply transform
            image = Image.fromarray(image.astype('uint8'))
            image = self.transform(image)
        
        label = torch.FloatTensor(label)
        return image, label

## PyTorch preprocessing (equivalent to Keras preprocess_input)
transform = transforms.Compose([
    transforms.Resize(100),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                       std=[0.229, 0.224, 0.225])
])

## Define model class
class BasicModel(nn.Module):
    def __init__(self, num_classes=3):
        super(BasicModel, self).__init__()
        # Load pretrained ResNet-50
        resnet = models.resnet50(pretrained=True)
        # Remove final FC layer (keep until avgpool)
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        # GlobalAveragePooling is already included in ResNet's avgpool
        self.fc = nn.Linear(2048, num_classes)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        x = self.features(x)        # [batch, 2048, 1, 1]
        x = torch.flatten(x, 1)     # [batch, 2048]
        x = self.fc(x)              # [batch, 3]
        x = self.sigmoid(x)         # [batch, 3]
        return x

## load the ResNet model
resnet_model_base = BasicModel(num_classes=3).to(device)

## compile the model (PyTorch doesn't need compile, just define optimizer and loss)
criterion = nn.BCELoss()  # Binary Cross-Entropy Loss
optimizer = optim.Adam(resnet_model_base.parameters(), lr=1e-3)

In [ ]:
# model selection
## train-validation split
np.random.seed(2020)
n_img = x_dat.shape[0]
loc_train = np.random.choice(n_img, int(n_img * 0.8), replace=False)
loc_val = np.setdiff1d(np.arange(n_img), loc_train)
## start model training
time_start = datetime.now()
print('Start training:', time_start)
## This cell is for testing, actual training is in next cell
# Uncomment below to test:
# train_dataset_test = TreePatchDataset(x_dat[loc_train], y_dat[loc_train], transform=transform)
# train_loader_test = DataLoader(train_dataset_test, batch_size=128, shuffle=True)
# for epoch in range(1):  # Test 1 epoch
#     for images, labels in train_loader_test:
#         images = images.to(device)
#         labels = labels.to(device)
#         outputs = resnet_model_base(images)
#         loss = criterion(outputs, labels)
#         break  # Test only one batch
#     break
print('Time for model training:', datetime.now()-time_start)

In [ ]:
# retrain the model with selected parameters
## build the model
model = BasicModel(num_classes=3).to(device)
## compile the model (define optimizer and loss function)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()  # Binary Cross-Entropy Loss

## Create DataLoader
train_dataset = TreePatchDataset(x_dat, y_dat, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
val_dataset = TreePatchDataset(x_dat[loc_val], y_dat[loc_val], transform=transform)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=2)

## start model training
time_start = datetime.now()
print('Start training:', time_start)
np.random.seed(2020)
torch.manual_seed(2020)

num_epochs = 50
for epoch in range(num_epochs):
    # Training phase
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
        # Calculate accuracy (for multi-label: threshold at 0.5)
        predicted = (outputs > 0.5).float()
        train_correct += (predicted == labels).sum().item()
        train_total += labels.numel()
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            
            # Calculate accuracy
            predicted = (outputs > 0.5).float()
            val_correct += (predicted == labels).sum().item()
            val_total += labels.numel()
    
    # Print statistics (every epoch)
    train_loss_avg = train_loss / len(train_loader)
    train_acc = train_correct / train_total
    val_loss_avg = val_loss / len(val_loader)
    val_acc = val_correct / val_total
    
    print(f'Epoch [{epoch+1}/{num_epochs}], '
          f'Train Loss: {train_loss_avg:.4f}, Train Acc: {train_acc:.4f}, '
          f'Val Loss: {val_loss_avg:.4f}, Val Acc: {val_acc:.4f}')

## save the model
save_path = path_wd + 'output/models/Basic_' + str(THRESH_DIAM) + '.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': num_epochs,
    'train_loss': train_loss_avg,
    'val_loss': val_loss_avg
}, save_path)
print('Time for model training:', datetime.now()-time_start)